Prachee Prasad, 381060

Lab 3: Fine-tune GPT or GPT-2 for creative story generation.

In [1]:
# Install HuggingFace libraries
!pip install -q transformers datasets accelerate

In [2]:
#import libraries
import torch
from datasets import Dataset
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import DataCollatorForLanguageModeling
from transformers import Trainer, TrainingArguments

In [4]:
# Load pre-trained GPT-2 model and tokenizer
model_name = "gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

# GPT2 has no default padding token
tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [5]:
# Sample creative story dataset
stories = [
    "Once upon a time, a curious robot discovered a hidden library beneath the city.",
    "In a quiet village, a young girl found a magical key that opened doors to different worlds.",
    "A lonely astronaut drifting through space heard a mysterious signal from an unknown planet.",
    "Deep in the enchanted forest, animals gathered every night to tell stories under the moonlight.",
    "A time traveler accidentally changed history and had to fix the future before it disappeared."
]

dataset = Dataset.from_dict({"text": stories})

print(dataset)

Dataset({
    features: ['text'],
    num_rows: 5
})


In [6]:
# Tokenization function
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [7]:
#Data Collator - prepares data for language model training.
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False   # GPT models use causal language modeling
)

In [10]:
#Training Configuration
training_args = TrainingArguments(
    output_dir="./gpt2-story",
    # overwrite_output_dir=True, # This parameter is causing the error
    num_train_epochs=5,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=2,
    logging_steps=100
)

In [11]:
#Trainer Setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

In [12]:
#Fine-Tune GPT-2
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=15, training_loss=2.8988629659016927, metrics={'train_runtime': 61.5551, 'train_samples_per_second': 0.406, 'train_steps_per_second': 0.244, 'total_flos': 816537600000.0, 'train_loss': 2.8988629659016927, 'epoch': 5.0})

In [13]:
# Function to generate stories
def generate_story(prompt):

    inputs = tokenizer(prompt, return_tensors="pt")

    output = model.generate(
        **inputs,
        max_length=120,
        num_return_sequences=1,
        temperature=0.9,
        do_sample=True,
        top_k=50
    )

    story = tokenizer.decode(output[0], skip_special_tokens=True)
    print(story)

# Example prompt
generate_story("Once upon a time in a futuristic city")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Once upon a time in a futuristic city, a holographic device was left behind by a mysterious entity who sought to take on the world. The holographic device was used to communicate with the human race from the future, but were unable to travel on their true selves. To destroy the holographic device in order to restore the world, the holographic device was stolen by the human race. They had to find the holographic device, which they were unable to find, and did so, but were able to find the future. However, during a battle between the humans and the holographic device,


In [15]:
# Interactive Story Generator
while True:

    prompt = input("Enter story prompt (or type exit): ")

    if prompt.lower() == "exit":
        break

    generate_story(prompt)

Enter story prompt (or type exit): A princess named Prachee


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


A princess named Prachee, a young woman, was invited to join a mystical society by the goddess Alena. However, she was attacked by a mysterious creature and was trapped in a magical place called Mora. Prachee's mother was forced to hide out in her magical forest, and when it found her, she was banished to a hidden place. After some time, the mysterious creature came to the forest and made her memories, and had the memories from the forest created by Prachee. The new memories were sent to the forest, and Prachee returned back to the forest
Enter story prompt (or type exit): exit
